In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import IsolationForest

class TransactionIsoForestModel:
    def __init__(
        self,
        n_estimators=200,
        contamination="auto",
        max_samples="auto",
        random_state=42
    ):
        self.scaler = StandardScaler()
        self.model = IsolationForest(
            n_estimators=n_estimators,
            contamination=contamination,
            max_samples=max_samples,
            random_state=random_state,
            n_jobs=-1
        )

        self.encoders = {}
        self.fitted = False

    def _preprocess_data(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        cat_cols = ["customer", "gender", "merchant", "category", "zipcodeOri", "zipMerchant"]

        for col in cat_cols:
            if col not in self.encoders:
                le = LabelEncoder()
                df[col] = le.fit_transform(df[col].astype(str))
                self.encoders[col] = le
            else:
                df[col] = self.encoders[col].transform(df[col].astype(str))

        df["amount"] = df["amount"].astype(float)
        df["log_amount"] = np.log1p(df["amount"])

        return df.select_dtypes(include=[np.number])

    def fit(self, df_raw: pd.DataFrame):
        X = self._preprocess_data(df_raw)
        X_scaled = self.scaler.fit_transform(X)
        self.model.fit(X_scaled)
        self.fitted = True
        print("Model fitted on", X.shape[0], "transactions.")
        return self

    def predict(self, df_raw: pd.DataFrame) -> pd.DataFrame:
        if not self.fitted:
            raise ValueError("Model not fitted. Call fit() first.")

        X = self._preprocess_data(df_raw)
        X_scaled = self.scaler.transform(X)

        preds = self.model.predict(X_scaled)
        scores = self.model.decision_function(X_scaled)

        out = df_raw.copy()
        out["anomaly_label"] = preds
        out["anomaly_score"] = scores

        return out


In [3]:
df = pd.read_csv("../data/bs140513_032310.csv")

iso = TransactionIsoForestModel()
iso.fit(df)

results = iso.predict(df)
print(results.head())


Model fitted on 594643 transactions.
   step       customer  age gender zipcodeOri       merchant zipMerchant  \
0     0  'C1093826151'  '4'    'M'    '28007'   'M348934600'     '28007'   
1     0   'C352968107'  '2'    'M'    '28007'   'M348934600'     '28007'   
2     0  'C2054744914'  '4'    'F'    '28007'  'M1823072687'     '28007'   
3     0  'C1760612790'  '3'    'M'    '28007'   'M348934600'     '28007'   
4     0   'C757503768'  '5'    'M'    '28007'   'M348934600'     '28007'   

              category  amount  fraud  anomaly_label  anomaly_score  
0  'es_transportation'    4.55      0             -1      -0.028004  
1  'es_transportation'   39.68      0              1       0.042272  
2  'es_transportation'   26.89      0              1       0.051939  
3  'es_transportation'   17.25      0              1       0.037381  
4  'es_transportation'   35.72      0              1       0.024236  


In [4]:
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

results["predicted_fraud"] = results["anomaly_label"].map({-1:1, 1:0})

cm = confusion_matrix(results["fraud"], results["predicted_fraud"])
cm_df = pd.DataFrame(cm,
                     index=["Actual_NoFraud(0)", "Actual_Fraud(1)"],
                     columns=["Pred_NoFraud(0)", "Pred_Fraud(1)"])

print(cm_df)


                   Pred_NoFraud(0)  Pred_Fraud(1)
Actual_NoFraud(0)           496505          90938
Actual_Fraud(1)                  0           7200


In [5]:
print(classification_report(results["fraud"], results["predicted_fraud"]))


              precision    recall  f1-score   support

           0       1.00      0.85      0.92    587443
           1       0.07      1.00      0.14      7200

    accuracy                           0.85    594643
   macro avg       0.54      0.92      0.53    594643
weighted avg       0.99      0.85      0.91    594643



In [6]:
total_frauds = results["fraud"].sum()
caught_frauds = results[(results["fraud"]==1) & (results["predicted_fraud"]==1)].shape[0]

fraud_capture_rate = caught_frauds / total_frauds
print(f"Fraud Capture Rate: {fraud_capture_rate:.3f}")


Fraud Capture Rate: 1.000


In [7]:
normal_count = (results["fraud"]==0).sum()
false_positives = results[(results["fraud"]==0) & (results["predicted_fraud"]==1)].shape[0]

fpr = false_positives / normal_count
print(f"False Positive Rate: {fpr:.3f}")


False Positive Rate: 0.155
